In [ ]:
from google.colab import userdata

token = userdata.get('GH_TOKEN')
usuario = "Nancy523"
repo = "TT2"
repo_owner = "HarielPS"  # El dueño del repo original

!git clone https://{token}@github.com/{repo_owner}/{repo}.git
%cd {repo}

Cloning into 'TT2'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 110 (delta 29), reused 100 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 2.48 MiB | 23.96 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/TT2


In [ ]:
!git config --global user.name "Nancy523"
!git config --global user.email "aguilarnancy529@gmail.com "

In [ ]:
!git branch        # locales
!git branch -r     # remotas

* main
  origin/Fase1_v2
  origin/Fase_1
  origin/HEAD -> origin/main
  origin/main


In [ ]:
!git status


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
rama = "Fase1_v2"  # Cambia esto por el nombre de tu rama

!git fetch origin
!git checkout {rama}

# Verifica en qué rama estás
!git branch

Branch 'Fase1_v2' set up to track remote branch 'Fase1_v2' from 'origin'.
Switched to a new branch 'Fase1_v2'
* Fase1_v2
  main


In [ ]:
# Intenta empujar los cambios a la rama remota
!git push -u origin Fase_1_Nancy

error: src refspec Fase_1_Nancy does not match any
error: failed to push some refs to 'https://github.com/HarielPS/TT2'


In [ ]:
# Lista los archivos en el directorio actual (que debería ser TT2)
!ls

configs  data  notebooks  notes  outputs  README.md  requirements.txt  src


In [ ]:
# También puedes listar el contenido de subdirectorios, por ejemplo:
!ls notebooks/

01_setup_experiments.ipynb     05_full_cv_experiments_time.pdf
01_setup_experiments.pdf       06_nuevo.ipynb
02_run_small_batch.ipynb       06_nuevo.pdf
02_run_small_batch.pdf	       06_refinamiento_prompt_muestra_20.pdf
03_run_medium_batch.ipynb      07_exploracion_hiperparametros.ipynb
03_run_medium_batch.pdf        07_exploracion_hiperparametros.pdf
04_analyze_medium_batch.ipynb  08_exploracion_profunda.ipynb
04_analyze_medium_batch.pdf    08_exploracion_profunda.pdf
05_full_cv_experiments.ipynb   outputs


In [ ]:
!git add .
!git commit -m "Prueba-Conexion-Collab"

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
!dir

configs  data  notebooks  notes  outputs  README.md  requirements.txt  src


In [ ]:
!git checkout -b Fase1_v2 origin/Fase1_v2

Branch 'Fase1_v2' set up to track remote branch 'Fase1_v2' from 'origin'.
Switched to a new branch 'Fase1_v2'


In [ ]:
!git switch -c Fase1_

fatal: A branch named 'Fase1_v2' already exists.


In [ ]:
from pathlib import Path
import sys
from datetime import datetime

# PROJECT_ROOT debería apuntar a la raíz del repositorio, que es el directorio actual después de %cd TT2/
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ruleset_comparison"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"exp_ruleset_comparison_{RUN_TS}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("EXPERIMENT_ID:", EXPERIMENT_ID)


PROJECT_ROOT: /content/TT2
DATA_DIR: /content/TT2/data
OUTPUT_DIR: /content/TT2/outputs/ruleset_comparison
EXPERIMENT_ID: exp_ruleset_comparison_20260326_034918


In [ ]:
import json
import pandas as pd

from configs.models import MODELS
from configs.rules import RULESETS
from src.experiment.runner import ExperimentRunner
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

In [ ]:
SAMPLE_PATH = DATA_DIR / "Muestra_csv.csv"

df_sample = pd.read_csv(SAMPLE_PATH)

print("Shape:", df_sample.shape)
print("Columnas:", list(df_sample.columns))
display(df_sample.head(3))

Shape: (20, 17)
Columnas: ['id', 'idFinal', 'grupo', 'motivo', 'Segmento', 'Propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']


,id,idFinal,grupo,motivo,Segmento,Propuesta,idcod,atr0,atr1,atr2,atr3,atr4,atr5,atr6,atr7,atr8,lex
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,Vivian,10,1,4,5.0,NaN,NaN,NaN,NaN,NaN,1
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Vivian,14,5,1,6.0,NaN,NaN,NaN,NaN,NaN,1
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,Vivian,5,6,19,1.0,NaN,NaN,NaN,NaN,NaN,1


In [ ]:
META_COLS = ["idFinal", "grupo", "motivo", "lex"]

required_cols = ["id", "Segmento", "Propuesta"]
missing = [c for c in required_cols if c not in df_sample.columns]

if missing:
    raise ValueError(f"Faltan columnas requeridas en la muestra: {missing}")

df_refine = df_sample.copy()

df_refine = df_refine.rename(
    columns={
        "id": "sample_id",
        "Segmento": "source_text",
        "Propuesta": "reference_text",
    }
)

keep_cols = ["sample_id", "source_text", "reference_text"] + [c for c in META_COLS if c in df_refine.columns]
df_refine = df_refine[keep_cols].copy()

print("Shape refinamiento:", df_refine.shape)
display(df_refine.head(5))

Shape refinamiento: (20, 7)


,sample_id,source_text,reference_text,idFinal,grupo,motivo,lex
0,2088,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,2976,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,3430,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1
3,3679,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,1
4,3145,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,1


In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "source": """A continuación una tabla donde se demuestra como crecen los ahorros al pasar los años: Edad Inversión Intereses Saldo Inversión Intereses Saldo Laura Ganados Alberto Ganados 22 $800 $80 $880 23 $800 $168 $1.232 Valor al Jubilarse $311.092 Valor al Jubilarse $263.232 Menos las contribuciones iniciales $6.400 Menos las contribuciones iniciales $28.800 Ganancia Neta $304.692 Ganancia Neta $234.""",
        "target": """A continuación, presentamos una tabla donde se demuestra como crecen los ahorros al pasar los años. Las categorías en que se divide la tabla son edad, inversión, intereses ganados y saldo. La tabla contrasta las ganancias en intereses por año de Laura y Alberto con una proyección de edad de jubilación de sesenta y cinco años y según la cantidad de años que mantuvieron el ahorro."""
    },
    {
        "source": """El varias veces mencionado Yager, nos dice acertadamente sobre estos aspectos lo siguiente: La mayoría de ustedes están hambrientos de tener libertad financiera, están hastiados de tener batallas financieras porque son como un carrusel.""",
        "target": """Yager habla acertadamente sobre estos aspectos y dice que la mayoría ambiciona tener libertad financiera y está cansada de tener batallas financieras inútiles."""
    },
    {
        "source": """De acuerdo con esta medición, la proporción de pobres en la población mundial –quienes viven con menos de un dólar por día– descendió levemente entre 1987 y 1993, pues pasó del 30% al 29%.""",
        "target": """Según esta medición, la proporción de personas pobres en la población mundial descendió un poco entre mil novecientos ochenta y siete y mil novecientos noventa y tres. Es decir, el porcentaje de personas que vivían con menos de un dólar por día pasó del treinta al veintinueve por ciento."""
    },
    {
        "source": """- Rentabilidad
INTERÉS DE 2 PESOS SOBRE UNA INVERSIÓN DE 100 PESOS = RENTABILIDAD DE 2 % ((2 ÷ 100) × 100 = 2%) POR CADA 100 PESOS SE GANAN 2 PESOS
 Endeudamiento
 DEUDA DE 30 PESOS POR SOBRE EL CAPITAL DE 20 PESOS = ENDEUDAMIENTO DE 1,5 VECES EL CAPITAL (30 ÷ 20 = 1,5) POR CADA PESO DE CAPITAL EXISTE 1,5 PESOS DE DEUDA
 Liquidez
80 PESOS DE DINERO DISPONIBLE POR SOBRE DEUDAS DE 40 PESOS = LIQUIDEZ DE DOS VECES EL VALOR DE LA DEUDA (80 ÷ 40 = 2) POR CADA PESO DE DEUDA SE DISPONE DE 2 PESOS PARA PAGO""",
        "target": """EL INTERÉS DE DOS PESOS SOBRE UNA INVERSIÓN DE CIEN PESOS DA UNA RENTABILIDAD DE DOS POR CIENTO. ES DECIR, POR CADA CIEN PESOS GANAMOS DOS PESOS. LA DEUDA DE TREINTA PESOS SOBRE EL CAPITAL DE VEINTE PESOS RESULTA EN UNA DEUDA DE UNO PUNTO CINCO VECES EL CAPITAL. ES DECIR, POR CADA PESO DE CAPITAL HAY UNO PUNTO CINCO PESOS DE DEUDA. EN OCHENTA PESOS DE DINERO DISPONIBLE SOBRE DEUDAS DE CUARENTA PESOS DA UNA LIQUIDEZ DEL DOBLE DE LA DEUDA. POR CADA PESO ADEUDADO HAY DOS PESOS PARA PAGAR."""
    },
]

FEW_SHOT_EXAMPLE_IDS = [78, 1805, 3635, 5262]

print("Número de ejemplos few-shot:", len(FEW_SHOT_EXAMPLES))
print("IDs few-shot:", FEW_SHOT_EXAMPLE_IDS)

Número de ejemplos few-shot: 4
IDs few-shot: [78, 1805, 3635, 5262]


In [ ]:

PROMPT_TYPE = "few-shot"

FINALIST_CONFIGS = [
    {
        "model_key": "llama3",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
        "config_label": "llama3_cfg_1",
    },
    {
        "model_key": "llama3",
        "temperature": 0.7,
        "top_p": 0.90,
        "repetition_penalty": 1.1,
        "max_new_tokens": 512,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
        "config_label": "llama3_cfg_2",
    },
    {
        "model_key": "mistral",
        "temperature": 0.7,
        "top_p": 0.90,
        "repetition_penalty": 1.1,
        "max_new_tokens": 512,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
        "config_label": "mistral_cfg_1",
    },
    {
        "model_key": "mistral",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 400,
        "do_sample": True,
        "no_repeat_ngram_size": 4,
        "config_label": "mistral_cfg_2",
    },
]

ACTIVE_RULESETS = ["R0", "R1", "R2", "R3", "R4"]

print("Prompt type fijo:", PROMPT_TYPE)
print("N configuraciones finalistas:", len(FINALIST_CONFIGS))
print("Rulesets:", ACTIVE_RULESETS)
display(pd.DataFrame(FINALIST_CONFIGS))

Prompt type fijo: few-shot
N configuraciones finalistas: 4
Rulesets: ['R0', 'R1', 'R2', 'R3', 'R4']


,model_key,temperature,top_p,repetition_penalty,do_sample,no_repeat_ngram_size,config_label,max_new_tokens
0,llama3,0.3,0.9,1.15,True,4,llama3_cfg_1,NaN
1,llama3,0.7,0.9,1.10,True,4,llama3_cfg_2,512.0
2,mistral,0.7,0.9,1.10,True,4,mistral_cfg_1,512.0
3,mistral,0.3,0.9,1.15,True,4,mistral_cfg_2,400.0


In [ ]:
for cfg in FINALIST_CONFIGS:
    if cfg["model_key"] not in MODELS:
        raise ValueError(f"Modelo no definido en MODELS: {cfg['model_key']}")

for ruleset in ACTIVE_RULESETS:
    if ruleset not in RULESETS:
        raise ValueError(f"Ruleset no definido en RULESETS: {ruleset}")

if PROMPT_TYPE not in ["zero-shot", "few-shot"]:
    raise ValueError(f"Técnica no soportada: {PROMPT_TYPE}")

print("Configuración validada correctamente.")

Configuración validada correctamente.


In [ ]:
# Celda 1: Instalar dependencia faltante
!apt-get install -y zstd

# Celda 2: Ahora sí instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 45 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (747 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current us

In [ ]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(5)

try:
    r = requests.get("http://localhost:11434/api/version", timeout=5)
    print("Ollama iniciado ✓", r.json())
except:
    print("Error: Ollama no respondió")

Ollama iniciado ✓ {'version': '0.18.2'}


In [ ]:
# Celda 3 - Descargar Mistral
!ollama pull mistral:7b-instruct-v0.2-q4_0

In [ ]:
!ollama list

NAME                             ID              SIZE      MODIFIED      
mistral:7b-instruct-v0.2-q4_0    61e88e884507    4.1 GB    8 seconds ago    


In [ ]:

runner = ExperimentRunner(
    experiment_id=EXPERIMENT_ID,
    log_dir=str(PROJECT_ROOT / "outputs" / "logs")
)

print("Runner inicializado:", runner.experiment_id)

Runner inicializado: exp_ruleset_comparison_20260326_034918


In [ ]:
# Celda para descargar el modelo llama3
!ollama pull llama3:8b

In [ ]:
test_row = df_refine.iloc[0]
test_cfg = FINALIST_CONFIGS[1] # Changed from FINALIST_CONFIGS[0] to FINALIST_CONFIGS[1]

test_record = runner.run_one(
    dataset_name="muestra_20_ruleset_comparison",
    model_key=test_cfg["model_key"],
    prompt_type=PROMPT_TYPE,
    ruleset=ACTIVE_RULESETS[0],
    source_text=test_row["source_text"],
    reference_text=test_row["reference_text"],
    sample_id=str(test_row["sample_id"]),
    fold_id=None,
    split_name="ruleset_comparison",
    few_shot_examples=FEW_SHOT_EXAMPLES if PROMPT_TYPE == "few-shot" else None,
    few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if PROMPT_TYPE == "few-shot" else None,
    generation_config={
        "temperature": test_cfg["temperature"],
        "top_p": test_cfg["top_p"],
        "repetition_penalty": test_cfg["repetition_penalty"],
        "max_new_tokens": test_cfg["max_new_tokens"],
    },
)

test_record.to_dict()

{'experiment_id': 'exp_ruleset_comparison_20260326_034918',
 'run_id': 'bbadc7c6-b45f-4a4e-9ca6-ffdb133a7a44',
 'timestamp': '2026-03-26T03:56:57.621372',
 'dataset_name': 'muestra_20_ruleset_comparison',
 'fold_id': None,
 'split_name': 'ruleset_comparison',
 'model_key': 'llama3',
 'model_id': 'meta-llama/Meta-Llama-3-8B-Instruct',
 'backend': 'ollama',
 'prompt_type': 'few-shot',
 'ruleset': 'R0',
 'few_shot_example_ids': [78, 1805, 3635, 5262],
 'generation_config': {'temperature': 0.7,
  'top_p': 0.9,
  'repetition_penalty': 1.1,
  'max_new_tokens': 512},
 'sample_id': '2088',
 'source_text': 'La comunidad humana más antigua ha sido denominada horda primitiva.',
 'reference_text': 'La comunidad humana más antigua se llamó tribu.',
 'generated_text': 'La comunidad humana más antigua se conoce como horda primitiva.',
 'prompt_text': 'Reescribe en español cada texto con lenguaje más claro y sencillo.\nConserva el significado original y no inventes información.\n\nDevuelve solo la ver

In [ ]:
# Celda para descargar el modelo llama3
!ollama pull llama3:8b

In [26]:
all_records = []

total_runs = len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)
run_counter = 0

for cfg in FINALIST_CONFIGS:
    for ruleset in ACTIVE_RULESETS:
        for _, row in df_refine.iterrows():
            run_counter += 1

            generation_config = {
                "temperature": cfg["temperature"],
                "top_p": cfg["top_p"],
                "repetition_penalty": cfg["repetition_penalty"],
                "max_new_tokens": cfg.get("max_new_tokens", 512), # Use .get() with a default value
            }

            print(
                f"[{run_counter}/{total_runs}] "
                f"model={cfg['model_key']} | cfg={cfg['config_label']} | "
                f"ruleset={ruleset} | sample_id={row['sample_id']}"
            )

            record = runner.run_one(
                dataset_name="muestra_20_ruleset_comparison",
                model_key=cfg["model_key"],
                prompt_type=PROMPT_TYPE,
                ruleset=ruleset,
                source_text=row["source_text"],
                reference_text=row["reference_text"],
                sample_id=str(row["sample_id"]),
                fold_id=None,
                split_name="ruleset_comparison",
                few_shot_examples=FEW_SHOT_EXAMPLES if PROMPT_TYPE == "few-shot" else None,
                few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if PROMPT_TYPE == "few-shot" else None,
                generation_config=generation_config,
            )

            record_dict = record.to_dict()
            record_dict["config_label"] = cfg["config_label"]

            for meta_col in ["idFinal", "grupo", "motivo", "lex"]:
                if meta_col in row.index:
                    record_dict[meta_col] = row[meta_col]

            all_records.append(record_dict)

print(f"Corridas completadas: {len(all_records)}")

[1/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=2088
[2/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=2976
[3/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3430
[4/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3679
[5/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3145
[6/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=507
[7/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=1756
[8/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3093
[9/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3192
[10/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3525
[11/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3627
[12/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3206
[13/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=329
[14/400] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3481
[15/400] model=ll

### Guarda y Copia el Notebook

Primero, asegúrate de haber guardado tu notebook actual (por ejemplo, `MiNotebookParaSubir.ipynb`). Luego, ejecuta la siguiente celda para copiarlo a la carpeta `notebooks` de tu repositorio local.

In [5]:
from pathlib import Path
import sys
import os

# Define PROJECT_ROOT and other paths
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ruleset_comparison"

# Ensure OUTPUT_DIR exists, though not directly used in copy, it's good practice to have it
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# *** REEMPLAZA ESTO CON EL NOMBRE REAL DE TU NOTEBOOK ***
notebook_filename = "pruebas_parametros.ipynb"

# Ruta donde Colab guarda tu notebook por defecto o donde lo hayas guardado
source_path = os.path.join("/content/", notebook_filename)

# Ruta de destino dentro del repositorio clonado
dest_dir = PROJECT_ROOT / "notebooks"

# Asegúrate de que el directorio de destino exista
dest_dir.mkdir(parents=True, exist_ok=True)

dest_path = dest_dir / notebook_filename

# Copia el archivo
!cp "{source_path}" "{dest_path}"

print(f"Notebook '{notebook_filename}' copiado a {dest_path}")

cp: cannot stat '/content/pruebas_parametros.ipynb': No such file or directory
Notebook 'pruebas_parametros.ipynb' copiado a /content/notebooks/pruebas_parametros.ipynb


### Añade, Commitea y Sube el Notebook a GitHub

Ahora, utilizaremos Git para añadir, commitear y subir tu notebook a la rama `Fase1_v2` de tu repositorio en GitHub.

In [2]:
# Agrega el notebook copiado al área de preparación de Git
!git add notebooks/{notebook_filename}

# Confirma los cambios
!git commit -m "Agregado: {notebook_filename} a la carpeta notebooks"

# Sube los cambios a la rama remota 'Fase1_v2'
# Si es la primera vez que subes a esta rama desde Colab, usa '-u' para establecer el seguimiento
!git push -u origin Fase1_v2

print("Notebook subido exitosamente a GitHub en la rama Fase1_v2!")

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
Notebook subido exitosamente a GitHub en la rama Fase1_v2!


In [ ]:
results_df = pd.DataFrame(all_records)

print("Shape results_df:", results_df.shape)
display(results_df.head(3))

if "status" in results_df.columns:
    display(results_df["status"].value_counts(dropna=False))

NameError: name 'pd' is not defined

In [ ]:
raw_results_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_raw_results.csv"
results_df.to_csv(raw_results_path, index=False, encoding="utf-8-sig")

print(raw_results_path)

/content/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260325_041653_raw_results.csv


In [ ]:
successful_df = results_df[results_df["status"] == "success"].copy()

print("Corridas exitosas:", len(successful_df))
print("Corridas con error:", len(results_df) - len(successful_df))
display(successful_df.head(3))

Corridas exitosas: 400
Corridas con error: 0


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,prompt_text,inference_seconds,status,error_message,metrics,config_label,idFinal,grupo,motivo,lex
0,exp_ruleset_comparison_20260325_041653,f9f8a2fa-b2e1-473d-ba92-c9cc8bdb4b9c,2026-03-25T04:42:09.730744,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,4.7795,success,None,{},llama3_cfg_1,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_ruleset_comparison_20260325_041653,1f3432dd-8af9-45fa-99db-8bca38fc7c88,2026-03-25T04:42:14.860037,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,14.1274,success,None,{},llama3_cfg_1,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_ruleset_comparison_20260325_041653,c0614e0e-7d03-4f7b-b7f0-e65a7a77fc2a,2026-03-25T04:42:29.297120,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,4.3932,success,None,{},llama3_cfg_1,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


In [ ]:
evaluated_df = evaluate_dataframe(
    successful_df,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
)

print("Shape evaluated_df:", evaluated_df.shape)
display(evaluated_df.head(3))

Shape evaluated_df: (400, 52)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sbert_similarity
0,exp_ruleset_comparison_20260325_041653,f9f8a2fa-b2e1-473d-ba92-c9cc8bdb4b9c,2026-03-25T04:42:09.730744,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.222222,0.300000,52.468333,34.855000,17.613333,None,None,None,None,None
1,exp_ruleset_comparison_20260325_041653,1f3432dd-8af9-45fa-99db-8bca38fc7c88,2026-03-25T04:42:14.860037,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.684211,0.625000,86.187632,105.172500,-18.984868,None,None,None,None,None
2,exp_ruleset_comparison_20260325_041653,c0614e0e-7d03-4f7b-b7f0-e65a7a77fc2a,2026-03-25T04:42:29.297120,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.222222,0.176471,78.079444,76.229118,1.850327,None,None,None,None,None


In [ ]:
param_cols = evaluated_df["generation_config"].apply(pd.Series)
param_cols = param_cols.rename(columns=lambda c: f"gen_{c}")

evaluated_df = pd.concat([evaluated_df, param_cols], axis=1)

display(
    evaluated_df[
        [
            "model_key",
            "config_label",
            "ruleset",
            "gen_temperature",
            "gen_top_p",
            "gen_repetition_penalty",
            "gen_max_new_tokens",
        ]
    ].head(5)
)

,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens
0,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,512.0
1,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,512.0
2,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,512.0
3,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,512.0
4,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,512.0


In [ ]:
evaluated_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_evaluated.csv"
evaluated_df.to_csv(evaluated_path, index=False, encoding="utf-8-sig")

print(evaluated_path)

/content/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260325_041653_evaluated.csv


In [ ]:
summary_by_ruleset = summarize_metrics(
    evaluated_df,
    group_cols=[
        "model_key",
        "config_label",
        "ruleset",
        "gen_temperature",
        "gen_top_p",
        "gen_repetition_penalty",
        "gen_max_new_tokens",
    ],
)

display(
    summary_by_ruleset.sort_values(
        by=["model_key", "config_label", "sari"],
        ascending=[True, True, False]
    ).head(30)
)

summary_by_ruleset_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_summary_by_ruleset.csv"
summary_by_ruleset.to_csv(summary_by_ruleset_path, index=False, encoding="utf-8-sig")

print(summary_by_ruleset_path)

,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta
0,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,512.0,42.575762,81.539995,88.314342,-6.774347,0.797999,0.75,0.510675,0.0,0.372833,0.496697,68.318459,51.983615,16.334844
1,llama3,llama3_cfg_1,R1,0.3,0.9,1.15,512.0,42.402353,84.003186,88.314342,-4.311156,0.838395,0.60,0.479457,0.0,0.414949,0.523725,66.292279,51.983615,14.308664
3,llama3,llama3_cfg_1,R3,0.3,0.9,1.15,512.0,41.170953,80.639558,88.314342,-7.674784,0.779360,0.65,0.443918,0.0,0.428395,0.561079,67.788471,51.983615,15.804856
2,llama3,llama3_cfg_1,R2,0.3,0.9,1.15,512.0,40.553949,84.813687,88.314342,-3.500655,0.790170,0.80,0.481461,0.0,0.444544,0.570911,73.199931,51.983615,21.216316
4,llama3,llama3_cfg_1,R4,0.3,0.9,1.15,512.0,40.135310,81.231695,88.314342,-7.082647,0.754194,0.50,0.483285,0.0,0.411947,0.551069,66.466617,51.983615,14.483002
6,llama3,llama3_cfg_2,R1,0.7,0.9,1.10,512.0,42.593872,82.164103,88.314342,-6.150239,0.828303,0.60,0.493686,0.0,0.372533,0.487582,65.884299,51.983615,13.900684
8,llama3,llama3_cfg_2,R3,0.7,0.9,1.10,512.0,41.769782,81.138805,88.314342,-7.175537,0.777022,0.45,0.462291,0.0,0.436307,0.560296,64.857375,51.983615,12.873760
5,llama3,llama3_cfg_2,R0,0.7,0.9,1.10,512.0,41.703724,78.365761,88.314342,-9.948581,0.772876,0.30,0.489633,0.0,0.296100,0.477226,67.121796,51.983615,15.138181
7,llama3,llama3_cfg_2,R2,0.7,0.9,1.10,512.0,41.118085,84.084170,88.314342,-4.230172,0.755792,0.55,0.468630,0.0,0.350469,0.512872,69.575468,51.983615,17.591853
9,llama3,llama3_cfg_2,R4,0.7,0.9,1.10,512.0,41.094112,86.343560,88.314342,-1.970782,0.685128,0.35,0.500221,0.0,0.339515,0.531641,71.786676,51.983615,19.803062


/content/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260325_041653_summary_by_ruleset.csv


In [ ]:
best_ruleset_per_config = (
    summary_by_ruleset
    .sort_values(
        by=["model_key", "config_label", "sari"],
        ascending=[True, True, False]
    )
    .groupby(["model_key", "config_label"], as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_ruleset_per_config)

best_ruleset_per_config_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_best_ruleset_per_config.csv"
best_ruleset_per_config.to_csv(best_ruleset_per_config_path, index=False, encoding="utf-8-sig")

print(best_ruleset_per_config_path)

,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta
0,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,512.0,42.575762,81.539995,88.314342,-6.774347,0.797999,0.75,0.510675,0.0,0.372833,0.496697,68.318459,51.983615,16.334844
1,llama3,llama3_cfg_2,R1,0.7,0.9,1.10,512.0,42.593872,82.164103,88.314342,-6.150239,0.828303,0.60,0.493686,0.0,0.372533,0.487582,65.884299,51.983615,13.900684
2,mistral,mistral_cfg_1,R4,0.7,0.9,1.10,512.0,42.361980,84.668359,88.314342,-3.645984,1.621183,0.95,0.403148,0.0,0.516062,0.366814,58.573274,51.983615,6.589659
3,mistral,mistral_cfg_2,R0,0.3,0.9,1.15,400.0,40.351687,81.550883,88.314342,-6.763459,1.430590,0.75,0.445094,0.0,0.484046,0.362257,51.156103,51.983615,-0.827512


/content/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260325_041653_best_ruleset_per_config.csv


In [ ]:
best_ruleset_per_model = summarize_metrics(
    evaluated_df,
    group_cols=["model_key", "ruleset"]
)

display(
    best_ruleset_per_model.sort_values(
        by=["model_key", "sari"],
        ascending=[True, False]
    )
)

best_ruleset_per_model_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_best_ruleset_per_model.csv"
best_ruleset_per_model.to_csv(best_ruleset_per_model_path, index=False, encoding="utf-8-sig")

print(best_ruleset_per_model_path)

,model_key,ruleset,sari,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta
1,llama3,R1,42.498112,83.083644,88.314342,-5.230698,0.833349,0.600,0.486572,0.0,0.393741,0.505654,66.088289,51.983615,14.104674
0,llama3,R0,42.139743,79.952878,88.314342,-8.361464,0.785438,0.525,0.500154,0.0,0.334466,0.486962,67.720127,51.983615,15.736513
3,llama3,R3,41.470368,80.889182,88.314342,-7.425160,0.778191,0.550,0.453104,0.0,0.432351,0.560688,66.322923,51.983615,14.339308
2,llama3,R2,40.836017,84.448929,88.314342,-3.865413,0.772981,0.675,0.475046,0.0,0.397506,0.541892,71.387699,51.983615,19.404085
4,llama3,R4,40.614711,83.787628,88.314342,-4.526714,0.719661,0.425,0.491753,0.0,0.375731,0.541355,69.126647,51.983615,17.143032
9,mistral,R4,40.607201,84.322834,88.314342,-3.991509,1.635399,0.950,0.385814,0.0,0.533070,0.366986,57.034790,51.983615,5.051175
5,mistral,R0,40.395926,82.072402,88.314342,-6.241940,1.312077,0.825,0.447281,0.0,0.465168,0.373872,57.534414,51.983615,5.550800
6,mistral,R1,39.919488,84.185609,88.314342,-4.128734,1.721447,1.425,0.429513,0.0,0.527458,0.356614,58.829939,51.983615,6.846324
7,mistral,R2,39.874273,81.970756,88.314342,-6.343586,1.515745,0.875,0.426380,0.0,0.502177,0.360879,55.556216,51.983615,3.572602
8,mistral,R3,39.402124,81.829182,88.314342,-6.485160,1.519195,0.925,0.399024,0.0,0.497989,0.372681,54.949459,51.983615,2.965845


/content/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260325_041653_best_ruleset_per_model.csv


In [ ]:
selected_configs = best_ruleset_per_config[
    ["model_key", "config_label", "ruleset"]
].drop_duplicates()

finalist_cases = evaluated_df.merge(
    selected_configs,
    on=["model_key", "config_label", "ruleset"],
    how="inner"
)

qual_cols = [
    "sample_id",
    "idFinal",
    "grupo",
    "motivo",
    "model_key",
    "config_label",
    "ruleset",
    "gen_temperature",
    "gen_top_p",
    "gen_repetition_penalty",
    "source_text",
    "reference_text",
    "generated_text",
    "sari",
    "bertscore_f1",
    "rougeL_f",
    "compression_ratio_eval",
    "exact_copy",
]

qual_cols = [c for c in qual_cols if c in finalist_cases.columns]

qual_review_df = finalist_cases[qual_cols].copy()

display(qual_review_df.head(20))

qual_review_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_qualitative_review.csv"
qual_review_df.to_csv(qual_review_path, index=False, encoding="utf-8-sig")

print(qual_review_path)

,sample_id,idFinal,grupo,motivo,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,source_text,reference_text,generated_text,sari,bertscore_f1,rougeL_f,compression_ratio_eval,exact_copy
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,llama3_cfg_1,R0,0.3,0.9,1.15,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llama horda...,72.215608,None,None,0.900000,0
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Prueba a guardar cada día una moneda de un dól...,30.299705,None,None,1.187500,0
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,La asignación de precios a los productos es un...,39.028196,None,None,1.058824,0
3,3679,829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",El precio total del kilo de pan incluyendo el ...,50.566696,None,None,0.866667,0
4,3145,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,"Por cada cien dólares vendidos, la empresa gas...",54.497570,None,None,1.272727,0
5,507,562_LibroUAC_Sincopyright.pdf,B_medianos,Porcentajes y rentabilidad.,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,"En general, se dice que la rentabilidad normal...","Generalmente, la rentabilidad normal de una in...",La rentabilidad normal de una inversión en paí...,67.608388,None,None,0.750000,0
6,1756,5100_LibroBAC.pdf,B_medianos,Redacción abstracta/institucional.,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,En virtud de estas y otras reflexiones fue que...,Debido a estas y otras reflexiones se diseñó e...,Este es nuestro dinero.,27.149949,None,None,0.160000,0
7,3093,875_LibroUide_Sincopyright.pdf,B_medianos,Definición contable.,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,Las cuentas de activo reflejan aquello que pos...,Las cuentas de activo evidencian las pertenenc...,Las cuentas de activo muestran lo que una orga...,63.333333,None,None,1.000000,0
8,3192,1202_LibroUide_Sincopyright.pdf,B_medianos,"Relación de pasivo/activo, útil para precisión.",llama3,llama3_cfg_1,R0,0.3,0.9,1.15,"Por cada dólar de pasivo corriente, la compañí...",La empresa tiene noventa y tres centavos de dó...,"Por cada dólar de pasivo corriente, la empresa...",43.651147,None,None,0.842105,0
9,3525,230_LibroUAC_Sincopyright.pdf,B_medianos,Dato económico con cifra.,llama3,llama3_cfg_1,R0,0.3,0.9,1.15,Posee la renta per cápita más alta de Latinoam...,Tiene la renta per cápita más alta de Latinoam...,La renta per cápita en Latinoamérica es la más...,39.269551,None,None,0.904762,0


/content/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260325_041653_qualitative_review.csv


In [ ]:
bitacora = {
    "experiment_id": EXPERIMENT_ID,
    "dataset_name": "muestra_20_ruleset_comparison",
    "n_samples": int(len(df_refine)),
    "prompt_type_fixed": PROMPT_TYPE,
    "finalist_configs": FINALIST_CONFIGS,
    "rulesets_tested": ACTIVE_RULESETS,
    "expected_total_runs": int(len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)),
    "successful_runs": int(len(successful_df)),
    "leader_metrics": ["sari", "bertscore_f1"],
    "support_metrics": ["rougeL_f", "compression_ratio_eval", "exact_copy"],
    "best_ruleset_per_config": best_ruleset_per_config.to_dict(orient="records"),
}

bitacora_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_bitacora_experimento.json"

with open(bitacora_path, "w", encoding="utf-8") as f:
    json.dump(bitacora, f, ensure_ascii=False, indent=2)

print(bitacora_path)

/content/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260325_041653_bitacora_experimento.json


In [ ]:
!ls /content/TT2/outputs/ruleset_comparison/

exp_ruleset_comparison_20260322_133408_best_ruleset_per_config.csv
exp_ruleset_comparison_20260322_133408_best_ruleset_per_model.csv
exp_ruleset_comparison_20260322_133408_bitacora_experimento.json
exp_ruleset_comparison_20260322_133408_evaluated.csv
exp_ruleset_comparison_20260322_133408_qualitative_review.csv
exp_ruleset_comparison_20260322_133408_raw_results.csv
exp_ruleset_comparison_20260322_133408_summary_by_ruleset.csv
exp_ruleset_comparison_20260325_041653_best_ruleset_per_config.csv
exp_ruleset_comparison_20260325_041653_best_ruleset_per_model.csv
exp_ruleset_comparison_20260325_041653_bitacora_experimento.json
exp_ruleset_comparison_20260325_041653_evaluated.csv
exp_ruleset_comparison_20260325_041653_qualitative_review.csv
exp_ruleset_comparison_20260325_041653_raw_results.csv
exp_ruleset_comparison_20260325_041653_summary_by_ruleset.csv


In [ ]:
from pathlib import Path
import sys
from datetime import datetime

# PROJECT_ROOT debería apuntar a la raíz del repositorio, que es el directorio actual después de %cd TT2/
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ruleset_comparison"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True);

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"exp_comparison_{RUN_TS}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

PROJECT_ROOT: /content/TT2
DATA_DIR: /content/TT2/data
OUTPUT_DIR: /content/TT2/outputs/ruleset_comparison
EXPERIMENT_ID: exp_comparison_20260325_061420


In [ ]:
!zip -r TT2.zip TT2/

	zip warning: name not matched: TT2/

zip error: Nothing to do! (try: zip -r TT2.zip . -i TT2/)


In [ ]:
print(f"""
Resumen rapido de esta fase:

- Se probaron {len(FINALIST_CONFIGS)} configuraciones finalistas.
- Se fijo la tecnica en: {PROMPT_TYPE}
- Se compararon {len(ACTIVE_RULESETS)} rulesets: {ACTIVE_RULESETS}
- En total se esperaban {len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)} corridas.

El objetivo de esta fase fue ver como cambia el desempeño cuando se modifica el conjunto de reglas,
manteniendo fijas las configuraciones que ya habian sido seleccionadas en la etapa de hiperparametros.
La comparacion se hizo principalmente con SARI y BERTScore F1, apoyandose tambien en otras metricas
para revisar el balance entre simplificacion, preservacion del significado y nivel de compresion.
""")


Resumen rapido de esta fase:

- Se probaron 4 configuraciones finalistas.
- Se fijo la tecnica en: few-shot
- Se compararon 5 rulesets: ['R0', 'R1', 'R2', 'R3', 'R4']
- En total se esperaban 400 corridas.

El objetivo de esta fase fue ver como cambia el desempeño cuando se modifica el conjunto de reglas,
manteniendo fijas las configuraciones que ya habian sido seleccionadas en la etapa de hiperparametros.
La comparacion se hizo principalmente con SARI y BERTScore F1, apoyandose tambien en otras metricas
para revisar el balance entre simplificacion, preservacion del significado y nivel de compresion.

